# A Complete Guide to Data Preprocessing: Theory & Practice with Seaborn's Titanic Dataset

Data preprocessing is a critical first step in any machine learning pipeline. Raw data is often messy, incomplete, and inconsistent; preprocessing transforms it into a clean, structured format that models can learn from effectively. This guide covers the essential theory and provides a hands-on tutorial using a classic dataset.



## 1. Key Machine Learning Terminologies

Before diving into code, it's crucial to understand the fundamental building blocks of a machine learning project.

*   **Features (or Input Variables):** These are the measurable characteristics or properties of the data used to make predictions. In a tabular dataset, each column (except the target) is a feature. For example, a passenger's `age`, `fare`, and `sex` are features used to predict survival[reference:0].
*   **Label (or Target Variable):** This is the output variable we want the model to predict. It's the "answer" for each row of data. In our case, `survived` is the label[reference:1].
*   **Training Set:** A subset of the data used to fit (train) the machine learning model. The model learns the patterns and relationships between features and the label from this set[reference:2].
*   **Test Set:** A separate, held-out subset of data used to evaluate the final model's performance on unseen data. It provides an unbiased estimate of how the model will generalize[reference:3].
*   **Validation Set:** An optional third subset sometimes used during model development to tune hyperparameters and prevent overfitting to the training set[reference:4].


In [ ]:
# OneHotEncoder Example
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# Sample categorical data
X = np.array([['rat'], ['dog'], ['cat'], ['bird']])
print(X)
# Encoding the categorical variable
encoder = OneHotEncoder()
X_encoded = encoder.fit_transform(X)
X_encoded.toarray()

[['rat']
 ['dog']
 ['cat']
 ['bird']]


array([[0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [1., 0., 0., 0.]])

In [ ]:
# ColumnTransformer Example
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

# Sample data
X = np.array([[1, 'cat'], [2, 'dog'], [3, 'cat'], [4, 'bird']])
column_transformer = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), [0]),    # Apply StandardScaler to the first column
        ('cat', OneHotEncoder(), [1])       # Apply OneHotEncoder to the second column
    ]
)

# Transforming the data
X_transformed = column_transformer.fit_transform(X)
X_transformed

array([[-1.34164079,  0.        ,  1.        ,  0.        ],
       [-0.4472136 ,  0.        ,  0.        ,  1.        ],
       [ 0.4472136 ,  0.        ,  1.        ,  0.        ],
       [ 1.34164079,  1.        ,  0.        ,  0.        ]])

In [ ]:
# Pipeline Example
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Sample data
X = np.array([[np.nan, 1], [2, 3], [3, 4]])
y = np.array([0, 1, 0])

# Creating a pipeline
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier())
])

# Fitting the pipeline
pipeline.fit(X, y)

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('classifier', RandomForestClassifier())])

## 2. Practical Guide: Preprocessing the Titanic Dataset

We'll use Seaborn's built-in `titanic` dataset, which contains information about 891 passengers and is ideal for demonstrating common preprocessing tasks[reference:5].


### Step 1: Import Libraries and Load Data

In [ ]:
# Import essential libraries
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Scikit-learn modules for preprocessing
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


#### Load Dataset

In [ ]:
# Load the Titanic dataset
df = sns.load_dataset('titanic')

#### Basic Information

In [ ]:
print("Dataset shape:", df.shape)

Dataset shape: (891, 15)


In [ ]:
print("\nFirst 5 rows:")
df.head()


First 5 rows:


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
# Data types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


### Insight:
- The `.info()` method reveals the dataset has `891` rows and `15` columns.
- Key columns include `survived` (label), `pclass`, `sex`, `age`, `fare`, `embarked`, and `deck`. It also shows missing values in `age` (177), `embarked` (2), and `deck` (688)

### Step 2: Initial Data Exploration & Cleaning

In [ ]:
# Separate features (X) and label (y)
X = df.drop('survived', axis=1)
y = df['survived']

In [ ]:
# Check the distribution of the target variable
print("Survival distribution:\n", y.value_counts(normalize=True))


Survival distribution:
 survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


In [ ]:
# Analyze missing values
print("\nMissing values per column:")
X.isnull().sum()


Missing values per column:


pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

#### Insight:
The `deck` column has over 77% missing values (688 out of 891). Imputing this much missing data is unreliable, so we drop it. This is a common strategy when a feature has a very high proportion of missing values.


#### Handling missing value column

In [ ]:
# Handle the 'deck' column: too many missing values (>70%), often best to drop
X_clean = X.drop(columns=['deck'])
print(f"Dropped 'deck'. \nOld shape: {X.shape}\nNew shape: {X_clean.shape}")

Dropped 'deck'. 
Old shape: (891, 14)
New shape: (891, 13)


<p style="text-align:center; font-family: 'Lucida Console', 'Courier New', monospace; font-size: 40px; border-style: solid; padding: 15px 0; background-color:red; font-weight: 50px;">Expolartory Data Analysis By Your Own</p>


### Step 3: Splitting Data into Training & Test Sets

Always split your data *before* any fitting or transformation to prevent **data leakage**, where information from the test set inadvertently influences the training process.

In [ ]:
# Split data into training and test sets (80%/20% is common)
X_train, X_test, y_train, y_test = train_test_split(X_clean, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Training set size: (712, 13)
Test set size: (179, 13)


### Step 4: Defining Preprocessing Strategies

Different types of features require different processing. We'll use Scikit-learn's `ColumnTransformer` to apply specific transformations to different columns simultaneously.

In [ ]:
# check the df.describe

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [ ]:
# Define numeric and categorical columns
numeric_features = ['age', 'fare', 'sibsp', 'parch']
categorical_features = ['sex', 'embarked', 'class', 'who', 'adult_male', 'alone', 'embark_town']


In [ ]:
# Create preprocessing pipelines for each type
numeric_transformer = Pipeline(steps=[
    # Impute missing numeric values with the median (robust to outliers)
    ('imputer', SimpleImputer(strategy='median')),
    # Scale features to have zero mean and unit variance
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    # Impute missing categorical values with the most frequent value
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Convert categories into binary (0/1) columns
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine transformers into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    # Keep columns that are not explicitly transformed (like 'pclass', 'alive')
    remainder='passthrough'
)

**Explanation:**
*   **Imputation:** We fill in missing values. For numbers (`age`), we use the median. For categories (`embarked`), we use the most frequent value.
*   **Scaling:** Numeric features like `age` and `fare` have different scales. `StandardScaler` transforms them to a common scale (mean=0, standard deviation=1), which is crucial for algorithms like SVMs or logistic regression.
*   **Encoding:** Categorical features like `sex` (`male`/`female`) are converted into a binary format (e.g., `sex_male`, `sex_female`) so mathematical models can use them.

### Step 5: Applying the Preprocessing Pipeline

In [ ]:
# Fit the preprocessor on the training data and transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)  # Important: use transform, not fit_transform
print("Preprocessing complete.")
print("Processed training set shape:", X_train_processed.shape)
print("Processed test set shape:", X_test_processed.shape)

Preprocessing complete.
Processed training set shape: (712, 24)
Processed test set shape: (179, 24)


In [ ]:
# Get feature names after preprocessing
# (OneHotEncoder creates new columns, so we need to get the final names)
feature_names = preprocessor.get_feature_names_out()
feature_names.tolist()

['num__age',
 'num__fare',
 'num__sibsp',
 'num__parch',
 'cat__sex_female',
 'cat__sex_male',
 'cat__embarked_C',
 'cat__embarked_Q',
 'cat__embarked_S',
 'cat__class_First',
 'cat__class_Second',
 'cat__class_Third',
 'cat__who_child',
 'cat__who_man',
 'cat__who_woman',
 'cat__adult_male_False',
 'cat__adult_male_True',
 'cat__alone_False',
 'cat__alone_True',
 'cat__embark_town_Cherbourg',
 'cat__embark_town_Queenstown',
 'cat__embark_town_Southampton',
 'remainder__pclass',
 'remainder__alive']

In [ ]:
# Convert the processed arrays back to DataFrames for inspection
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

print("Processed Training Data Shape:", X_train_df.shape)

Processed Training Data Shape: (712, 24)


In [ ]:
print("Processed Training Data (First 5 rows):")
X_train_df.head()

Processed Training Data (First 5 rows):


,num__age,num__fare,num__sibsp,num__parch,cat__sex_female,cat__sex_male,cat__embarked_C,cat__embarked_Q,cat__embarked_S,cat__class_First,cat__class_Second,cat__class_Third,cat__who_child,cat__who_man,cat__who_woman,cat__adult_male_False,cat__adult_male_True,cat__alone_False,cat__alone_True,cat__embark_town_Cherbourg,cat__embark_town_Queenstown,cat__embark_town_Southampton,remainder__pclass,remainder__alive
0,-0.081135,0.513812,-0.465084,-0.466183,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,3,yes
1,-0.081135,-0.662563,-0.465084,-0.466183,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,2,no
2,-0.081135,3.955399,-0.465084,-0.466183,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1,no
3,-0.887827,-0.467874,-0.465084,0.727782,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,3,yes
4,0.110934,-0.115977,0.478335,0.727782,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,2,yes


In [ ]:
print("Missing values in processed training set:", X_train_df.isnull().sum().sum())

Missing values in processed training set: 0


### Step 6: Basic Feature Engineering (Creating New Features)

Feature engineering involves creating new features from existing ones to help the model uncover better patterns.

In [ ]:
# Create a new feature 'family_size'
X_train_df['family_size'] = X_train['sibsp'] + X_train['parch'] + 1
X_test_df['family_size'] = X_test['sibsp'] + X_test['parch'] + 1

# Create a binary feature 'is_child'
X_train_df['is_child'] = (X_train['age'] < 18).astype(int)
X_test_df['is_child'] = (X_test['age'] < 18).astype(int)


In [ ]:

# Binning (Discretization) - Convert continuous 'fare' into categories
# Binning groups numerical values into buckets, which can reveal non-linear relationships[reference:13].
fare_bins = [0, 10, 50, 100, 600]
fare_labels = ['Very_Low', 'Low', 'Medium', 'High']
X_train_df['fare_bin'] = pd.cut(X_train['fare'], bins=fare_bins, labels=fare_labels)
X_test_df['fare_bin'] = pd.cut(X_test['fare'], bins=fare_bins, labels=fare_labels)

# One-hot encode the new binned column
fare_bin_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
fare_bin_train_encoded = fare_bin_encoder.fit_transform(X_train_df[['fare_bin']])
fare_bin_test_encoded = fare_bin_encoder.transform(X_test_df[['fare_bin']])

# Add the new one-hot encoded columns to our DataFrames
fare_bin_columns = fare_bin_encoder.get_feature_names_out(['fare_bin'])
X_train_df[fare_bin_columns] = fare_bin_train_encoded
X_test_df[fare_bin_columns] = fare_bin_test_encoded

# Drop the intermediate 'fare_bin' column
X_train_df.drop(columns=['fare_bin'], inplace=True)
X_test_df.drop(columns=['fare_bin'], inplace=True)

print("Final Training Data Shape after Feature Engineering:", X_train_df.shape)

Final Training Data Shape after Feature Engineering: (712, 31)


## Conclusion

Your data is now preprocessed and ready for modeling. The key steps we covered were:
1.  **Understanding the data** and its flaws (missing values, incorrect types).
2.  **Splitting the data** to avoid leakage.
3.  **Imputing missing values** strategically.
4.  **Scaling numeric features** and **encoding categorical features**.
5.  **Engineering new features** to provide more signal to the model.

The final `X_train_df` and `X_test_df` DataFrames contain clean, normalized, and encoded data. You can now feed `X_train_df` and `y_train` into any Scikit-learn classifier (e.g., `RandomForestClassifier`, `LogisticRegression`) to build a model for predicting passenger survival.

Remember, preprocessing is not a one-size-fits-all process. The strategies shown here—median imputation, standard scaling, one-hot encoding—are common starting points. Always tailor these steps to the specific characteristics of your dataset and the requirements of your chosen machine learning algorithm.

(179, 31)
